Initialize models

In [ ]:
from transformers import AutoModelForSequenceClassification, DebertaV2Tokenizer

model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)
tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-base")

Prepare dataset

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("csv", data_files="./Language Challenge/quora-insincere-questions-classification/train.csv")['train']
def preprocesss_function(example):
    text = [str(x) for x in example['question_text']]

    model_data = tokenizer(text, truncation=True, max_length=512)
    model_data['labels']=example['target']
    return model_data

train_dataset = train_dataset.map(preprocesss_function, batched=True, remove_columns=train_dataset.column_names)
train_dataset



In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Training the model

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="./sentence-safety-deberta",
    overwrite_output_dir=False,
    learning_rate=2e-5,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint=True)

Evaluating the F1 Score

In [ ]:
import torch
device = torch.device('cuda')
model = model.to(device)

from torch.utils.data import DataLoader

model.eval()

train_dataset.set_format('torch')
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
true_pos, false_pos, false_neg = 0,0,0
iterate=0
with torch.no_grad():
    for batch in train_loader:
        iterate+=1;
        inputs = {
            "input_ids": batch['input_ids'].to(device),
            "attention_mask": batch['attention_mask'].to(device)
        }
        output = model(**inputs).logits
        logit = torch.argmax(output, dim=-1)

        labels = batch['labels'].to(device)

        true_pos += ((logit == 1) & (labels == 1)).sum().item()
        false_pos += ((logit == 1) & (labels == 0)).sum().item()
        false_neg += ((logit == 0) & (labels == 1)).sum().item()
        

precision = true_pos/(true_pos + false_pos)
recall = true_pos/(true_pos + false_neg)
print(2*precision*recall/(precision + recall))